# Agent Management — End-to-End Quickstart

This notebook showcases the `agent_management` library as a **complete, functional product** for managing
Snowflake Cortex Agents and Semantic Views through a governed lifecycle.

It walks the full path a new project would take:

```
config  ->  validate  ->  semantic views  ->  agents  ->  evals  ->  testing
```

Every step maps to a product CLI (`agent-mgmt-*`) and to a matching GitHub Actions workflow, so this
notebook doubles as living documentation of the CI/CD flow.

## How to run it

- It is **environment-parameterized**: set `ENV = "dev"` or `ENV = "prod"` in the Parameters cell.
- It defaults to **`LIVE = False`** (dry-run / read-only). A top-to-bottom run never mutates Snowflake.
- Set `LIVE = True` to actually deploy + evaluate against the selected environment. Live cells are clearly marked with **[LIVE]**.

### Lifecycle

```mermaid
flowchart LR
    cfg[Config] --> val[Validate]
    val --> sv[Semantic Views]
    sv --> ag[Agents]
    ag --> ev[Evals]
    ev --> test[Testing]
```

## 1. Install the library

The library is pip-installable. Console scripts (`agent-mgmt-*`) only resolve once the package is installed,
which is exactly what CI does via the `snowflake-setup` composite action.

Run the install cell once (uncomment if your environment does not already have it).

In [ ]:
# Install the package in editable mode so the agent-mgmt-* console scripts resolve.
# Uncomment the next line the first time you run this notebook in a fresh environment.
# %pip install -e ..

import agent_management

print("agent_management version:", agent_management.__version__)

## 2. Parameters

- `ENV` — `"dev"` or `"prod"`. Controls which database/role/warehouse the commands target (resolved from `project.yml`).
- `LIVE` — `False` keeps everything dry-run / read-only. `True` performs real deploys and evals.

For **live** runs you must also have Snowflake credentials available to the connector, e.g. set
`SNOWFLAKE_CONNECTION_NAME` (named connection) or `SNOWFLAKE_ACCOUNT` / `SNOWFLAKE_USER` /
`SNOWFLAKE_PRIVATE_KEY_PATH` in your environment before launching Jupyter.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

# ---- Parameters -------------------------------------------------------------
ENV = "dev"      # "dev" or "prod"
LIVE = False     # False = dry-run/read-only; True = real deploys + evals
# -----------------------------------------------------------------------------

# Run commands from the repository root (parent of notebooks/).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def sh(cmd: str) -> int:
    """Run a shell command from the repo root and stream its output."""
    print(f"$ {cmd}\n")
    proc = subprocess.run(
        shlex.split(cmd), cwd=REPO_ROOT, capture_output=True, text=True
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0 and proc.stderr:
        print(proc.stderr)
    print(f"[exit {proc.returncode}]")
    return proc.returncode


def live_or_dry(base_cmd: str, dry_flag: str = "--dry-run") -> int:
    """Run base_cmd live when LIVE is True, otherwise append the dry-run flag."""
    return sh(base_cmd if LIVE else f"{base_cmd} {dry_flag}")


print(f"ENV={ENV}  LIVE={LIVE}  REPO_ROOT={REPO_ROOT}")

## 3. Configuration

The library reads project-wide defaults from `project.yml` and resolves per-environment settings
(database, role, warehouse, schemas, name suffix, aliases) from the `environments/` configs.

`load_project_config()` and `load_env_config()` also accept explicit paths (or the
`AGENT_MGMT_PROJECT_CONFIG` / `AGENT_MGMT_ENV_CONFIG` env vars) so the package works after a plain
`pip install`, without needing to run from this reference repo.

In [ ]:
from agent_management.utils.config import load_env_config, load_project_config

# Ensure config discovery resolves relative to the repo root.
os.environ.setdefault("AGENT_MGMT_PROJECT_CONFIG", str(REPO_ROOT / "project.yml"))

project = load_project_config()
env_cfg = load_env_config(env=ENV)

print(f"Project: {project['project']['name']}  v{project['project']['version']}\n")
print(f"Environment: {ENV}")
print(f"  database        : {env_cfg['deployment']['database']}")
print(f"  semantic schema : {env_cfg['deployment']['semantic_schema']}")
print(f"  agents schema   : {env_cfg['deployment']['agents_schema']}")
print(f"  role            : {env_cfg['snowflake']['role']}")
print(f"  warehouse       : {env_cfg['snowflake']['warehouse']}")

print("\nConfigured agents:")
for name, spec in project.get("agents", {}).items():
    print(f"  - {name}: {len(spec.get('semantic_views', []))} semantic views")

## 4. Validate

Validation is fully local and read-only: it lints every agent spec and semantic-view YAML and renders
the Jinja templates for the target environment. This is the first gate in `validate-pr.yml`.

CLI: `agent-mgmt-validate --env <env>` and `agent-mgmt-validate-spec-format <spec> ...`

In [ ]:
# Validate all specs + SV YAMLs for the selected env (local, no Snowflake needed).
sh(f"agent-mgmt-validate --env {ENV}")

# Validate the agent spec format (tool-description template conventions).
sh("agent-mgmt-validate-spec-format agents/specs/resort_executive.yml agents/specs/ski_ops_assistant.yml")

## 5. Semantic Views

Semantic views are the data contract agents query. This project manages them via dbt (the
`semantic_views.source: dbt` setting), and the library handles VQR (verified query) sync and drift
detection.

CLIs:
- `agent-mgmt-sync-vqrs` — merge verified queries into the dbt SV models
- `agent-mgmt-deploy-svs --env <env>` — deploy / verify semantic views (`--dry-run` to preview)
- `agent-mgmt-detect-sv-drift --env <env>` — compare deployed SVs against source of truth

**[LIVE]** `deploy-svs` without `--dry-run` and `detect-sv-drift` both connect to Snowflake.

In [ ]:
# Preview the verified-query sync into dbt SV models (no writes when --dry-run).
sh("agent-mgmt-sync-vqrs --dry-run")

# Deploy / verify semantic views. Dry-run by default; real deploy when LIVE=True.
live_or_dry(f"agent-mgmt-deploy-svs --env {ENV}")

# Drift detection connects to Snowflake, so only run it in LIVE mode.
if LIVE:
    sh(f"agent-mgmt-detect-sv-drift --env {ENV}")
else:
    print("[skipped] agent-mgmt-detect-sv-drift requires a live Snowflake connection (set LIVE=True)")

## 6. Agents

Agents are deployed through the Cortex Agent Versioning lifecycle:

```
ADD LIVE VERSION FROM LAST  ->  MODIFY LIVE VERSION SET SPECIFICATION  ->  COMMIT  ->  set ALIAS
```

Aliases by environment: DEV uses `latest`; PROD uses `validated` (pre-customer) and `production`
(customer traffic). A `DEFAULT` alias is also maintained so selectorless REST calls resolve.

CLIs:
- `agent-mgmt-deploy-agents --env <env>` — deploy via the versioning path (`--dry-run` to preview the rendered spec)
- `agent-mgmt-smoke-agent --env <env> --alias <alias>` — post-deploy smoke test

We also preview the rendered spec with the Python API (`build_spec`) without any Snowflake call.

In [ ]:
# [LIVE] Smoke-test the deployed agent against the env's deploy alias.
# Requires a live deployment, so only runs when LIVE=True.
deploy_alias = env_cfg.get("agent", {}).get("deploy_alias", "latest")

if LIVE:
    sh(f"agent-mgmt-smoke-agent --env {ENV} --alias {deploy_alias}")
else:
    print(f"[skipped] agent-mgmt-smoke-agent --env {ENV} --alias {deploy_alias} (set LIVE=True)")

## 7. Evaluations

Two distinct evaluation tools share helper patterns but target different objects:

- **Semantic View evals** (`agent-mgmt-eval-sv`) — run `EXECUTE_AI_EVALUATION` over verified queries to score SQL correctness.
- **Agent evals** (`agent-mgmt-eval-agent`) — run the agent against golden question datasets and score answer correctness / logical consistency.

Both support `--dry-run` to render the eval plan without executing. Scoring/inventory helpers:
`agent-mgmt-sv-scores`, `agent-mgmt-metrics`, `agent-mgmt-check-sv-evals`.

**[LIVE]** Removing `--dry-run` runs real evaluations against Snowflake (this consumes warehouse credits and can take minutes).

In [ ]:
# Semantic View evaluation. Dry-run shows the eval config; LIVE runs EXECUTE_AI_EVALUATION.
if LIVE:
    sh(f"agent-mgmt-eval-sv --env {ENV} --run-name QUICKSTART-{ENV}")
else:
    sh(f"agent-mgmt-eval-sv --env {ENV} --dry-run")

print("\n" + "=" * 60 + "\n")

# Agent evaluation. Dry-run renders configs and shows the plan; LIVE runs the eval.
live_or_dry(f"agent-mgmt-eval-agent --env {ENV}")

## 8. Testing

The library ships a fast test suite. Running a focused subset here proves the package is healthy and
that the product layout / public API contracts hold.

In [ ]:
# Fast, offline checks: package layout, public API surface, and template rendering.
sh("python -m pytest tests/test_package_layout.py tests/test_public_api_surface.py "
   "tests/test_templates.py tests/test_smoke.py -q")

## 9. Recap — CLI to workflow map

Each lifecycle step maps to a product command and the CI/CD workflow that runs it:

| Step | Product command | Runs in workflow |
|---|---|---|
| Validate specs | `agent-mgmt-validate`, `agent-mgmt-validate-spec-format` | `validate-pr.yml` |
| Sync verified queries | `agent-mgmt-sync-vqrs` | `deploy-dev.yml`, `deploy-prod-validated.yml` |
| Deploy semantic views | `agent-mgmt-deploy-svs` | `deploy-dev.yml`, `deploy-prod-validated.yml` |
| Detect SV drift | `agent-mgmt-detect-sv-drift`, `agent-mgmt-detect-drift` | `validate-pr.yml` |
| Deploy agents | `agent-mgmt-deploy-agents` | `deploy-dev.yml`, `deploy-prod-validated.yml` |
| Smoke test agent | `agent-mgmt-smoke-agent` | `deploy-*`, `promote-validated-to-production.yml`, `rollback.yml` |
| Evaluate semantic views | `agent-mgmt-eval-sv`, `agent-mgmt-sv-scores` | `validate-pr.yml`, `deploy-dev.yml` |
| Evaluate agents | `agent-mgmt-eval-agent`, `agent-mgmt-metrics` | `validate-pr.yml`, `deploy-*` |
| Promote alias | `agent-mgmt-agent-versioning promote` | `promote-validated-to-production.yml` |
| Rollback | `agent-mgmt-rollback` | `rollback.yml` |

### Promotion flow

```mermaid
flowchart LR
    feat[feature branch] -->|PR| dev[dev: deploy-dev]
    dev -->|PR| main[main: deploy-prod-validated]
    main -->|manual| prod[promote-validated-to-production]
    prod -.->|on failure| rb[rollback]
```

This notebook ran the full lifecycle in **dry-run** by default. Set `LIVE = True` and re-run to deploy
and evaluate against `dev` or `prod`.

In [ ]:
from agent_management.agents import resolve_agent_identity

# Show how an agent's deployed identity resolves for the selected environment
# (the env name_suffix is applied automatically). No Snowflake call.
for short_name in project.get("agents", {}):
    demo_agent = {"metadata": {"name": short_name}}
    name, schema_fqn, fqn = resolve_agent_identity(demo_agent, env_cfg)
    print(f"{short_name:>20}  ->  {fqn}")

print()

# Deploy agents via the versioning path. Dry-run renders the spec and shows the
# plan; LIVE=True performs the ADD/MODIFY/COMMIT/alias lifecycle.
live_or_dry(f"agent-mgmt-deploy-agents --env {ENV}")